In [4]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm, trange


In [5]:
data = "./data/titanic_clean.csv"
df = pd.read_csv(data)
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 889 entries, 0 to 888
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    int64  
 1   Survived     889 non-null    int64  
 2   Pclass       889 non-null    int64  
 3   Name         889 non-null    str    
 4   Sex          889 non-null    str    
 5   Age          889 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Ticket       889 non-null    str    
 9   Fare         889 non-null    float64
 10  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.5 KB


In [6]:
%%time
df["gender"] = np.where(df["Sex"]=="male", 1,0)


CPU times: user 1.48 ms, sys: 476 μs, total: 1.95 ms
Wall time: 3.65 ms


In [7]:
%%time

#using iterrows

for i,row in df.iterrows():
    if(row["Sex"]=="male"):
        df.loc[i,"gender"] = 1
    else:
        df.loc[i,"gender"] = 0


CPU times: user 215 ms, sys: 4 ms, total: 219 ms
Wall time: 222 ms


In [8]:
df["gender"] = None

In [9]:
%%time
#using itertuples


for row in df.itertuples():
    if(row.Sex=="male"):
        df.at[row.Index,"gender"] = 1
    else:
        df.at[row.Index,"gender"] = 0




CPU times: user 20.5 ms, sys: 1.49 ms, total: 22 ms
Wall time: 21.4 ms


In [10]:
%%time
# using pandas

df.loc[df["Sex"]=="male","gender"] = 1
df.loc[df["Sex"]!="male","gender"] = 0


CPU times: user 1.36 ms, sys: 99 μs, total: 1.46 ms
Wall time: 1.37 ms


In [11]:
df["gender"]= np.nan



In [12]:
%%time 

df["gender"]= df["Sex"].apply(lambda x: 1 if x=="male" else 0)


CPU times: user 733 μs, sys: 41 μs, total: 774 μs
Wall time: 746 μs


In [13]:
df["gender"] = None 
conditions = [
    (df["Sex"]=="male"),]
choices = [1]

In [14]:
df["gender"]= None

In [15]:

%%time
df["gender"] = np.select(conditions,choices,default=0) 


CPU times: user 333 μs, sys: 123 μs, total: 456 μs
Wall time: 806 μs


In [16]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,gender
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,1


In [17]:
from  pandarallel import pandarallel as pdp
pdp.initialize(progress_bar=True, nb_workers=4)
df.head()



INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,gender
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,1


In [18]:
%%time
df["gender"] = df.parallel_apply(lambda x: 1 if x["Sex"]=="male" else 0, axis=1)

CPU times: user 80.1 ms, sys: 52.7 ms, total: 133 ms
Wall time: 283 ms


In [19]:
tqdm.pandas()

In [20]:
%%time
import time

for i in trange(10):
    time.sleep(0.1)
    pass

100%|██████████| 10/10 [00:01<00:00,  9.70it/s]

CPU times: user 25.6 ms, sys: 20.3 ms, total: 45.9 ms
Wall time: 1.08 s


In [21]:
%%time
df["gender"] = None
df["gender"] = df.progress_apply(lambda x: 1 if x["Sex"]=="male" else 0, axis=1)

100%|██████████| 889/889 [00:00<00:00, 72008.35it/s]

CPU times: user 13.2 ms, sys: 3.66 ms, total: 16.9 ms
Wall time: 17.9 ms
